In [35]:
import sys

sys.path.append("../src")

from preprocessing import clean_test_pipeline, get_feature_columns, clean_test_rolling_pipeline,clean_train_rolling_pipeline
from training import build_error_analysis,evaluate_model,cmapss_score

from config import ROLLING_SENSORS, ROLLING_WINDOWS

from xgboost import Booster,DMatrix


## Test model v1

In [36]:
test = clean_test_pipeline(
    "../data/test_FD001.txt",
    "../data/RUL_FD001.txt"
)
test.shape

(13096, 27)

In [37]:
model = Booster()

model.load_model(
    "../models/final_xgboost.json"
)

print("Model loaded successfully.")

Model loaded successfully.


In [38]:
last_cycles = (
    test.groupby("unit_number")["time_cycles"]
    .max()
    .reset_index()
)

last_cycles.head()

,unit_number,time_cycles
0,1,31
1,2,49
2,3,126
3,4,106
4,5,98


In [39]:
test_last = (
    test.sort_values(["unit_number", "time_cycles"])
    .groupby("unit_number")
    .tail(1)
    .sort_values("unit_number")
    .reset_index(drop=True)
)

test_last.shape

(100, 27)

In [40]:
test_last[
    ["unit_number", "time_cycles", "RUL"]
].head(10)

,unit_number,time_cycles,RUL
0,1,31,112
1,2,49,98
2,3,126,69
3,4,106,82
4,5,98,91
5,6,105,93
6,7,160,91
7,8,166,95
8,9,55,111
9,10,192,96


In [41]:
feature_cols = get_feature_columns(test)

X_test = test_last[feature_cols]

X_test.shape

(100, 18)

In [42]:
import xgboost as xgb

dtest = xgb.DMatrix(X_test)

y_pred_test = model.predict(dtest)

y_pred_test[:10]

array([180.7235  , 146.04031 ,  52.25793 ,  84.483246, 101.74184 ,
       104.12921 , 103.65062 ,  75.97629 , 136.53003 , 103.48982 ],
      dtype=float32)

In [43]:
test_results=build_error_analysis(test_last,y_pred_test)
test_results.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
0,1,31,112,180.723495,68.723495,68.723495
1,2,49,98,146.040314,48.040314,48.040314
2,3,126,69,52.257931,-16.742069,16.742069
3,4,106,82,84.483246,2.483246,2.483246
4,5,98,91,101.741837,10.741837,10.741837
5,6,105,93,104.129211,11.129211,11.129211
6,7,160,91,103.650620,12.650620,12.650620
7,8,166,95,75.976288,-19.023712,19.023712
8,9,55,111,136.530029,25.530029,25.530029
9,10,192,96,103.489822,7.489822,7.489822


In [44]:
from training import evaluate_model

test_metrics = evaluate_model(
    test_results["RUL"],
    test_results["predicted_RUL"],
    model_name="Final XGBoost - Test"
)

Final XGBoost - Test
MAE : 18.317594528198242
RMSE: 25.293157878153902
R²  : 0.6295355558395386


In [45]:
print("Mean error:", test_results["error"].mean())
print("MAE:", test_results["absolute_error"].mean())
print("Max error:", test_results["error"].max())
print("Min error:", test_results["error"].min())

Mean error: 10.148746418952943
MAE: 18.31759422779083
Max error: 74.72929382324219
Min error: -38.52690124511719


In [46]:
worst_test_errors = test_results.sort_values(
    "absolute_error",
    ascending=False
)

worst_test_errors.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
14,15,76,83,157.729294,74.729294,74.729294
0,1,31,112,180.723495,68.723495,68.723495
66,67,71,77,142.453171,65.453171,65.453171
77,78,72,107,164.330399,57.330399,57.330399
21,22,39,111,168.168076,57.168076,57.168076
49,50,74,79,130.182968,51.182968,51.182968
1,2,49,98,146.040314,48.040314,48.040314
26,27,140,66,112.769958,46.769958,46.769958
84,85,34,118,161.523788,43.523788,43.523788
13,14,46,107,150.313431,43.313431,43.313431


In [67]:
score_v2 = cmapss_score(
    test_results["RUL"],
    test_results["predicted_RUL"]
)

print("C-MAPSS Score v2:", score_v2)

C-MAPSS Score v2: 5228.43520582715


## Test model v2

In [47]:
test_rolling = clean_test_rolling_pipeline(
    "../data/test_FD001.txt",
    "../data/RUL_FD001.txt",
    sensors=ROLLING_SENSORS,
    windows=ROLLING_WINDOWS,
)

print("Test shape:", test_rolling.shape)

Test shape: (13096, 43)


In [48]:
train_rolling = clean_train_rolling_pipeline(
    "../data/train_FD001.txt",
    sensors=ROLLING_SENSORS,
    windows=ROLLING_WINDOWS,
)

feature_cols = get_feature_columns(
    train_rolling
)

print("Number of features:", len(feature_cols))

Number of features: 34


In [49]:
test_last = (
    test_rolling
    .sort_values(
        ["unit_number", "time_cycles"]
    )
    .groupby("unit_number")
    .tail(1)
    .sort_values("unit_number")
    .copy()
)

print("Shape:", test_last.shape)


Shape: (100, 43)


In [50]:
print(
    test_last[
        ["unit_number", "time_cycles", "RUL"]
    ].head(10)
)

      unit_number  time_cycles  RUL
30              1           31  112
79              2           49   98
205             3          126   69
311             4          106   82
409             5           98   91
514             6          105   93
674             7          160   91
840             8          166   95
895             9           55  111
1087           10          192   96


In [51]:
model = Booster()

model.load_model(
    "../models/final_xgboost_v2.json"
)

print("Model v2 loaded successfully.")

Model v2 loaded successfully.


In [52]:
X_test = test_last[feature_cols]

print("X_test shape:", X_test.shape)

X_test shape: (100, 34)


In [53]:
dtest = DMatrix(X_test)

y_pred_test = model.predict(dtest)

print("Number of predictions:", len(y_pred_test))
print("First 10 predictions:", y_pred_test[:10])

Number of predictions: 100
First 10 predictions: [179.51118  148.77138   57.07169   83.67181   99.29215   99.193565
 114.3158    93.27625  140.13058  110.42509 ]


In [54]:
test_results_v2 = build_error_analysis(
    test_last,
    y_pred_test
)

test_results.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
0,1,31,112,180.723495,68.723495,68.723495
1,2,49,98,146.040314,48.040314,48.040314
2,3,126,69,52.257931,-16.742069,16.742069
3,4,106,82,84.483246,2.483246,2.483246
4,5,98,91,101.741837,10.741837,10.741837
5,6,105,93,104.129211,11.129211,11.129211
6,7,160,91,103.650620,12.650620,12.650620
7,8,166,95,75.976288,-19.023712,19.023712
8,9,55,111,136.530029,25.530029,25.530029
9,10,192,96,103.489822,7.489822,7.489822


In [55]:
test_metrics_v2 = evaluate_model(
    test_results_v2["RUL"],
    test_results_v2["predicted_RUL"],
    model_name="XGBoost v2 - Test"
)

XGBoost v2 - Test
MAE : 19.029977798461914
RMSE: 25.619833225300034
R²  : 0.6199041604995728


In [56]:
worst_test_v2 = (
    test_results_v2
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(10)
)

worst_test_v2

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
8785,67,71,77,147.711884,70.711884,70.711884
1704,15,76,83,150.521622,67.521622,67.521622
30,1,31,112,179.511185,67.511185,67.511185
10153,78,72,107,166.428116,59.428116,59.428116
79,2,49,98,148.771378,50.771378,50.771378
3201,27,140,66,116.676430,50.676430,50.676430
6338,50,74,79,128.181717,49.181717,49.181717
2621,22,39,111,159.162659,48.162659,48.162659
4712,37,121,21,67.877289,46.877289,46.877289
12226,93,244,85,40.488937,-44.511063,44.511063


In [57]:
print("Mean Error:", test_results_v2["error"].mean())
print("Mean Absolute Error:", test_results_v2["absolute_error"].mean())
print("Max Error:", test_results_v2["error"].max())
print("Min Error:", test_results_v2["error"].min())

Mean Error: 12.47013017654419
Mean Absolute Error: 19.02997980117798
Max Error: 70.71188354492188
Min Error: -44.51106262207031


In [58]:
score_v2 = cmapss_score(
    test_results_v2["RUL"],
    test_results_v2["predicted_RUL"]
)

print("C-MAPSS Score v2:", score_v2)

C-MAPSS Score v2: 4740.877876657272


## Test with CAP

In [59]:
model_v3 = Booster()
model_v3.load_model("../models/final_xgboost_v3.json")

In [60]:
test_last_cap = (
    test_rolling
    .sort_values(["unit_number", "time_cycles"])
    .groupby("unit_number")
    .tail(1)
    .sort_values("unit_number")
    .copy()
)

print("Test last shape:", test_last_cap.shape)

Test last shape: (100, 43)


In [61]:
X_test_cap = test_last_cap[feature_cols]

print("X_test:", X_test_cap.shape)

X_test: (100, 34)


In [62]:
y_pred_cap_test = model_v3.predict(
    DMatrix(test_last_cap[feature_cols])
)

print("Number of predictions:", len(y_pred_cap_test))
print("First 10 predictions:", y_pred_cap_test[:10])

Number of predictions: 100
First 10 predictions: [123.540474 119.62893   52.806484  83.5059    94.48232   91.87541
 100.94067   87.15795  117.84017   93.844215]


In [63]:
test_results_3 = build_error_analysis(
    test_last_cap,
    y_pred_cap_test
)

test_results_3.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
30,1,31,112,123.540474,11.540474,11.540474
79,2,49,98,119.628929,21.628929,21.628929
205,3,126,69,52.806484,-16.193516,16.193516
311,4,106,82,83.505898,1.505898,1.505898
409,5,98,91,94.482323,3.482323,3.482323
514,6,105,93,91.875412,-1.124588,1.124588
674,7,160,91,100.940666,9.940666,9.940666
840,8,166,95,87.157951,-7.842049,7.842049
895,9,55,111,117.840172,6.840172,6.840172
1087,10,192,96,93.844215,-2.155785,2.155785


In [64]:
test_metrics = evaluate_model(test_results_3["RUL"], test_results_3["predicted_RUL"], model_name="XGBoost v3 - Test")

XGBoost v3 - Test
MAE : 13.478078842163086
RMSE: 18.117798946667747
R²  : 0.8099134564399719


In [65]:
print("Mean Error:", test_results_3["error"].mean())
print("Mean Absolute Error:", test_results_3["absolute_error"].mean())
print("Max Error:", test_results_3["error"].max())
print("Min Error:", test_results_3["error"].min())

Mean Error: 0.9758381366729736
Mean Absolute Error: 13.478078298568725
Max Error: 45.42427062988281
Min Error: -46.59558868408203


In [66]:
score = cmapss_score(
    test_results_3["RUL"],
    test_results_3["predicted_RUL"]
)

print("C-MAPSS Score:", score)

C-MAPSS Score: 618.8690153744714
